# BigData News Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davi7071/BigData-News-Analysis/blob/main/bigdata.ipynb)

Coleta títulos de portais de notícias brasileiros, processa o texto com PySpark (limpeza, stopwords, contagem de palavras e bigramas) e gera JSON, nuvem de palavras e gráficos de barras.

O código do pipeline fica em `news_analysis.py`; este notebook apenas executa as etapas e exibe os resultados.

## 1. Ambiente
No Colab, clona o repositório, instala as dependências e monta o Google Drive. Localmente, os resultados vão para `output/`.

In [ ]:
import sys

EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    ![ -d /content/BigData-News-Analysis ] || git clone -q https://github.com/Davi7071/BigData-News-Analysis.git /content/BigData-News-Analysis
    %cd /content/BigData-News-Analysis
    !pip install -q -r requirements.txt
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_SAIDA = "/content/drive/MyDrive/bigdata-news"
else:
    PASTA_SAIDA = "output"

## 2. Parâmetros

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt

import news_analysis as na

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

MAX_ARTIGOS = 30   # títulos por site
DIAS = None        # ex.: 7 para descartar artigos publicados há mais de uma semana
TOP_N = 20         # termos exportados no JSON

sites = na.NEWS_SITES  # edite aqui para usar outros portais
Path(PASTA_SAIDA).mkdir(parents=True, exist_ok=True)

## 3. Verificação dos sites

In [ ]:
sites_validos = na.filtrar_sites_validos(sites)

## 4. Coleta dos títulos
Apenas os sites válidos são raspados. Títulos repetidos (mesma URL ou mesmo texto) são removidos e os dados brutos são salvos em CSV para permitir reproduzir a análise.

In [ ]:
df_titulos = na.coletar(sites_validos, max_artigos=MAX_ARTIGOS, dias=DIAS)
caminho_csv = na.salvar_titulos(df_titulos, PASTA_SAIDA)
print("CSV salvo em:", caminho_csv)

df_titulos.groupby("site").size().sort_values(ascending=False)

## 5. Processamento no Spark
Limpeza com `regexp_replace` (URLs, pontuação, números e emojis), tokenização com `RegexTokenizer`, remoção de stopwords com `StopWordsRemover` e bigramas com `NGram`.

In [ ]:
spark = na.criar_spark()
stopwords = na.carregar_stopwords()

sdf = na.processar(spark, df_titulos, stopwords).cache()
sdf.select("titulo", "tokens").show(5, truncate=80)

## 6. Contagem de palavras e bigramas

In [ ]:
palavras = na.contar(sdf, "tokens").limit(200).toPandas()
bigramas = na.contar(sdf, "bigramas").limit(200).toPandas()

palavras.head(TOP_N)

In [ ]:
bigramas.head(TOP_N)

## 7. Exportação

In [ ]:
print("Palavras:", na.salvar_json(palavras.head(TOP_N), Path(PASTA_SAIDA) / "resultado.json"))
print("Bigramas:", na.salvar_json(bigramas.head(TOP_N), Path(PASTA_SAIDA) / "bigramas.json"))

## 8. Visualizações
Todos os gráficos usam a mesma contagem do Spark exportada no JSON.

In [ ]:
from datetime import datetime

data = datetime.now().strftime("%d/%m/%Y")
na.grafico_nuvem(palavras, f"Termos mais frequentes nos títulos ({data})", Path(PASTA_SAIDA) / "nuvem.png")
plt.show()

In [ ]:
na.grafico_barras(palavras.head(10), "Top 10 palavras mais frequentes", Path(PASTA_SAIDA) / "top10_palavras.png")
plt.show()

In [ ]:
na.grafico_barras(bigramas.head(10), "Top 10 bigramas mais frequentes", Path(PASTA_SAIDA) / "top10_bigramas.png")
plt.show()